# Thread Pool Executor
The `ThreadPoolExecutor` class is the most modern and easiest way to convert a for-loop to run concurrently for IO-bound tasks.

Thread pools are a design pattern that let you execute and manage heterogeneous, discrete, and ad hoc tasks.

In [2]:
import threading
from threading import Thread
from concurrent.futures import ThreadPoolExecutor, TimeoutError, wait, as_completed, FIRST_COMPLETED
from time import sleep
from random import random

# from urlib.request import urlopen
# from urlib.error import URLError, HTTPError
# from http import HTTPStatus

## Threads, Executors, and Thread Pools
A thread pool is a programming pattern for automatically managing pool of worker threads.

The pool is responsible for a fixed number of threads.
* It controls when they are created, such as when they are needed.
* It controls how many tasks each worker can execute before being replaced.
* It also controls what workers should do when they are not being used, such as making them wait without consuming computational resources.

Each thread in the pool is called a worker or a worker thread. Each worker is agnostic to the type of tasks that are executed, along with the user of the thread pool to execute a suite of similar (homogeneous) or dissimilar tasks (heterogeneous) in terms of the function called, function arguments, task duration, and more.

`concurrent.futures` module was introduced with Python 3.2 written by Brian Quinlan and provides both thread pools and process pools.

The `ThreadPoolExecutor` extends the `Executor` class and will return `Future` objects when it is called.
* `Executor`: Parent class for the `ThreadPoolExecutor` that defines basic life-cycle operations for the pool.
* `Future`: Object returned when submitting tasks to the thread pool that may complete later. It represents a delayed result for an async task. Also sometimes called a `promise` or `delay`.

`Executor` methods:
* `submit()`: Dispatch a function to be executed and return a `Future` object.
* `map()`: Call a function for each item in an iterable.
* `shutdown()`: Shut down the `Executor`.

`Future` methods:
* `cancelled()`: Returns `True` if the task was cancelled before being executed. 
* `running()`: Returns `True` if the task is currently running. 
* `done()`: Returns `True` if the task has completed or was canceled.
* `result()`: Access the result from running the task.
* `exception()`: Access any exception raised while running the task.
* `add_done_callback()`: Add a callback function to the task to be executed by the thread pool once the task is completed.

3 main steps in the life-cycle of using the `ThreadPoolExecutor`:
1. Create: Create the thread pool by calling the constructor `ThreadPoolExecutor()`.
2. Execute: Run tasks using workers via the `map()` or `submit()` methods.
3. Shut down: Shut down the thread pool by calling `shutdown()`.

We can process the results from these async tasks as the tasks are completed using the `as_completed()` module function.

A prefered way to work with the `ThreadPoolExecutor` class is to use a context manager interface.

In [3]:
# example running a function in the thread pool
def task():
    # report a message
    print("This is another thread")

# protect the entry point
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor() as exe:
        # issue the task
        future = exe.submit(task)
        # wait for the task to finish
        future.result()
    # close the thread pool automatically

This is another thread


Use `ThreadPoolExecutor` when:
* Your tasks can be defined by a pure function that has no state or side effects.
* Your task can fit within a single Python function, likely making it simple and easy to understand.
* You need to perform the same task many times with different arguments, e.g., homogeneous tasks.
* You need to call the same function for each object in a collection in a for-loop.

## Configure the `ThreadPoolExecutor`
* `max_workers`: Maximum number of worker threads to use in the pool. We will likely have many more thread workers than we have physical or logical CPU cores. This is because threads are lightweigth units of concurrency and we often require and easily support tens, hundreds, or even thousands of concurrent threads on modern systems. Use 100 or the number of tasks as a starting point and go from there.
* `thread_name_prefix`: The prefix assigned to each worker thread name.
* `initializer`: Function executed after each worker thread is created. Prior to executing the tasks.
* `initargs`: Arguments to the worker thread initialization function.

Default Worker Threads = min(32, Logical CPUs + 4). Note: CPUs with hyperthreading.

In [3]:
# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    exe = ThreadPoolExecutor()
    # report the status of the thread pool
    print(exe._max_workers)
    # shutdown the thread pool
    exe.shutdown()

12


In [4]:
# custom task function executed in the thread pool
def task(number):
    # block for a moment
    sleep(1)
    # report a message
    if number % 100 == 0:
        print(f"> task {number} done")
        
# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor(100) as exe:
        # issue many tasks to the pool
        _ = exe.map(task, range(1000))


> task 0 done
> task 100 done
> task 200 done
> task 300 done
> task 400 done
> task 500 done
> task 600 done
> task 700 done
> task 800 done
> task 900 done


In [5]:
# custom task function executed in the thread pool
def task(number):
    # block for a moment
    sleep(1)
    
# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor(4, thread_name_prefix="Downloader") as exe:
        # issue many tasks to the pool
        _ = exe.map(task, range(20))
        # report all thread names
        for thread in threading.enumerate():
            print(thread)

<_MainThread(MainThread, started 12256)>
<Thread(Thread-6 (_thread_main), started daemon 20520)>
<Heartbeat(Thread-7, started daemon 11700)>
<ControlThread(Thread-5, started daemon 19512)>
<HistorySavingThread(IPythonHistorySavingThread, started 9020)>
<ParentPollerWindows(Thread-4, started daemon 9776)>
<Thread(Downloader_0, started 19000)>
<Thread(Downloader_1, started 18152)>
<Thread(Downloader_2, started 20828)>
<Thread(Downloader_3, started 17660)>


In [6]:
# custom function to be executed in a worker thread
def task(number):
    # report a message
    print(f"Worker executing task {number}...")
    # block for a moment
    sleep(1)
    
# initialize a worker in the thread pool
def init():
    # report a message
    print("Initializing worker...")
    
# protect the entry point 
if __name__ == "__main__":
    # create and configure the thread pool
    with ThreadPoolExecutor(2, initializer=init) as exe:
        # issue tasks to the thread pool
        _ = exe.map(task, range(4))

Initializing worker...
Worker executing task 0...
Initializing worker...
Worker executing task 1...
Worker executing task 2...Worker executing task 3...



## Execute Multiple Tasks Concurrently
**1. Execute tasks with one argument**

In [8]:
# custom function to be executed in a worker thread
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor(4) as exe:
        # issue tasks to execute concurrently
        for result in exe.map(task, range(10)):
            # report results
            print(result)

Task generated 0.6538800279648195
Task generated 0.6021661257718037
Task generated 0.20407701863624872
Task generated 0.7275500244082599
Task generated 0.2249503388575126
Task generated 0.7814967557010101
Task generated 0.3981926132929694
Task generated 0.0495461481989809550.6538800279648195

1.6021661257718036
2.204077018636249
Task generated 0.7144455042780878
Task generated 0.86125542559206913.7275500244082598

4.2249503388575125
5.78149675570101
6.398192613292969
7.049546148198981
8.714445504278087
9.86125542559207


**2. Execute tasks with multiple arguments**
```python
exe.map(task, items1, items2)
```

In [3]:
# custom function to be executed in a worker thread
def task(number, value):
    # report a message
    print(f"Task using {value}")
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor(4) as exe:
        # prepare random numbers between 0 and 1
        values = [random() for _ in range(10)]
        # issue tasks to execute concurrently
        for result in exe.map(task, range(10), values):
            # report results
            print(result)

Task using 0.013593050007220442
Task using 0.04360255551944059
Task using 0.4196353766659735
Task using 0.9927780236466565
Task using 0.115921148317285280.013593050007220442

Task using 0.104350794235120731.0436025555194406

Task using 0.7290693927370244
Task using 0.20898561699731522
Task using 0.2602664576242111
Task using 0.092802104780766562.4196353766659735

3.9927780236466566
4.115921148317286
5.1043507942351205
6.729069392737024
7.208985616997316
8.260266457624212
9.092802104780766


**3. Execute tasks with no return values**

In [6]:
# custom function to be executed in a worker thread
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(value)
    
# protect the entry point 
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor(4) as exe:
        # issue tasks to execute concurrently
        _ = exe.map(task, range(10))
    # wait automatically for all tasks to finish ...

Task generated 0.7508132085781086
Task generated 0.14666829485624555
Task generated 0.3517535019196286
Task generated 0.4844324834415916
Task generated 0.16373414150305787
Task generated 0.36913835234071635
Task generated 0.6742476434576591
Task generated 0.7348177391171441
Task generated 0.48800640526484285
Task generated 0.7433023974830227


**4. Execute tasks with a Timeout**

We do not have to block forever when waiting for the return value from each task issued with the `map()` method. In fact, waiting forever is not so good practice and instead we should use a timeout wherever possible.

In [7]:
# custom function to be executed in a worker thread
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(number + value)
    # return a new value
    return number + value

# protect the entry points
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor(4) as exe:
        try:
            # issue tasks to execute concurrently
            for result in exe.map(task, range(10), timeout=2):
                # report results
                print(result)
        except TimeoutError:
            print("Gave up, took too long")
    # report that we will wait for the tasks to complete
    print("Waiting for tasks to complete ...")

Task generated 0.8442697073628763
Task generated 0.2909318738271415
Task generated 0.8927518863080224
Task generated 0.4389644571257969
Task generated 0.156342006416047560.8442697073628763

Task generated 0.403012036353561861.2909318738271414

Gave up, took too long
Waiting for tasks to complete ...


## Execute One-Off Tasks Asynchronously
The `Future` object is a promise to return the results from the task (if any) and provides a way to determine if a specific task has been completed or not.

**1. Execute a Task with One Argument**

In [9]:
# custom function to be executed in a worker thread
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # issue an asynchronous task
        future = exe.submit(task, 100)
        # get the result once the task completes
        result = future.result()
        # report the result
        print(result)

Task generated 0.34930912656734314
100.34930912656735


**2. Execute a Task with Multiple Arguments**

In [10]:
# custom function to be executed in a worker thread
def task(number, value):
    # report a message
    print(f"Task received {value}")
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor() as exe:
        # issue an asynchronous task
        future = exe.submit(task, 100, random())
        # get the result once the task completes
        result = future.result()
        # report the result
        print(result)

Task received 0.9842799613940736
100.98427996139408


**3. Execute a Task with No Arguments**

In [11]:
# custom function to be executed in a worker thread
def task():
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return value

# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # issue an asynchronous task
        future = exe.submit(task)
        # get the result once the task completes
        result = future.result()
        # report the result
        print(result)

Task generated 0.31312306981681637
0.31312306981681637


**4. Execute a Task with No Return Value**

Although the `submit()` method returns a `Future` object as a handle on an asynchronous task, we do not have to use it. It can be ignored and the caller can continue on with other tasks.

In [12]:
# custom function to be executed in a worker thread
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(value)
    
# protect the entry point
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor() as exe:
        # issue an asynchronous task
        _ = exe.submit(task, 100)
    # wait for all tasks to finish

Task generated 0.4274829935312098


**5. Execute Many One-Off Tasks**

In [13]:
# custom function to be executed in a worker thread
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}")
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the thread pool
    with ThreadPoolExecutor() as ex:
        # issue many asynchronous tasks systematically
        futures = [ex.submit(task, i) for i in range(5)]
        # enumerate futures and report results
        for future in futures:
            print(future.result)

Task generated 0.5088638915842661
Task generated 0.5820460887116926
Task generated 0.7703960592633002
Task generated 0.6801442047698109
Task generated 0.04832792442585865
<bound method Future.result of <Future at 0x1518bacbdf0 state=running>>
<bound method Future.result of <Future at 0x1518bb98f40 state=running>>
<bound method Future.result of <Future at 0x1518bb98be0 state=running>>
<bound method Future.result of <Future at 0x1518cd30d60 state=running>>
<bound method Future.result of <Future at 0x1518cd4b040 state=running>>


## Query Asynchronous Tasks
**1. Check status of `Future` objects**

A `Future` object can exist in one of three states:
1. Scheduled (pre-running)
2. Running
3. Done (post-running).

While a task is running, it can raise an uncaught exception, causing the execution of the task to stop. The exception will be stored and can be retrieved directly or will be re-raised if the result is attempted to be retrieved.

A *cancelled* task is always be in the *done* state.

In [3]:
# task function executed in a worker thread
def work():
    # block for a moment
    sleep(0.5)
    
# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # start one thread
        future = exe.submit(work)
        # confirm that the task is running
        running = future.running()
        done = future.done()
        print(f"Future running={running}, done={done}")
        # wait for the task to complete
        future.result()
        # confirm that the task is done
        running = future.running()
        done = future.done()
        print(f"Future running={running}, done={done}")

Future running=True, done=False
Future running=False, done=True


**2. Get results from `Future` objects**

Retrieve the result from the task by calling the `result()` method.

In [5]:
# task function executed in a worker thread
def work():
    # block for a moment
    sleep(1)
    # return a message
    return "All done"

# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # start one thread
        future = exe.submit(work)
        # get the result from the task, blocks
        result = future.result()
        # report the result
        print(f"Got Result: {result}")

Got Result: All done


It is good practice to limit how long we are willing to wait for a result.

In [7]:
# task function executed in a worker thread
def work():
    # block for a moment
    sleep(1)
    # return a message
    return "All done"

# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # start one thread
        future = exe.submit(work)
        try:
            # get the result from the task, blocks
            result = future.result(timeout=0.5)
            # report the result
            print(f"Got Result: {result}")
        except TimeoutError:
            print("Gave up waiting for a result")

Gave up waiting for a result


**3. Cancel `Future` Objects**
While a task is in the queue and before it has been started, we can cancel it by calling the `cancel()` method.

In [14]:
# custom function executed in a worker thread
def work(sleep_time):
    # block for a moment
    sleep(sleep_time)
    
# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor(1) as exe:
        # start a long running task
        future1 = exe.submit(work, 2)
        running = future1.running()
        print(f"First task running={running}")
        # start a second. future2 is queued because we are working with only 1 thread
        future2 = exe.submit(work, 0.1)
        running = future2.running()
        print(f"Second task running={running}")
        # cancel the second task
        canceled = future2.cancel()
        print(f"Second task was cancelled: {canceled}")

First task running=True
Second task running=False
Second task was cancelled: True


**4. Add callbacks to `Future` objects**

Register a callback function to be called once the task has completed by using `add_done_callback()` method.

Remember a task is completed if it finishes normally, if it is canceled or if an exception is raised.

In [15]:
# callback function to call when a task is completed
def custom_callback(future):
    # report message
    print(f"Custom callback got: {future.result()}")
    
# custom task function executed in a worker thread
def work():
    # block for a moment
    sleep(1)
    # return a result
    return 99

# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # execute the task 
        fut = exe.submit(work)
        # add the custom callback
        fut.add_done_callback(custom_callback)

Custom callback got: 99


**5. Get exceptions from `Future` objects**

In [16]:
# custom task function executed by a worker thread
def work():
    # block a moment
    sleep(1)
    # raise an exception
    raise Exception("Something bad happened!")
    
# protect the entry point
if __name__ == "__main__":
    # create a thread pool
    with ThreadPoolExecutor() as exe:
        # execute our task
        future = exe.submit(work)
        # wait for the task to be done
        wait([future])
        # get the exception
        exception = future.exception()
        print(f"Task exception={exception}")
        # get the result from the task
        try:
            result = future.result()
        except Exception:
            print("Unable to get the result")

Task exception=Something bad happened!
Unable to get the result
